In [1]:
import os
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

In [5]:
# Paths
base_dir = Path("../..")
json_path = "groups_same_second.json"
img_dir = base_dir / "data" / "floorplan_reoriented"
txt_dir = base_dir / "annotation" / "human_annotated_tags"
out_dir = Path(".") / "examples_output"
out_dir.mkdir(exist_ok=True)

# Load JSON entries
with open(json_path, 'r') as f:
    entries = json.load(f)

# Prepare font for subtitles
try:
    font = ImageFont.truetype("arial.ttf", size=16)
except IOError:
    font = ImageFont.load_default()

subtitle_height = 20  # space for ID subtitles

for idx, entry in enumerate(entries, start=1):
    group_ids = entry.get("group", [])
    images = []

    # Load images for this group
    for img_id in group_ids:
        img_path = img_dir / f"{img_id}.png"
        if img_path.exists():
            images.append((img_id, Image.open(img_path)))
        else:
            raise FileNotFoundError(f"Image file not found: {img_path}")

    # Determine layout
    widths, heights = zip(*(im.size for _, im in images))
    total_width = sum(widths)
    max_height = max(heights)

    # Create canvas
    canvas = Image.new('RGB', (total_width, max_height + subtitle_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    # Paste images and draw subtitles
    x_offset = 0
    for img_id, im in images:
        canvas.paste(im, (x_offset, 0))
        # Compute text size
        if hasattr(font, 'getsize'):
            text_w, text_h = font.getsize(img_id)
        else:
            bbox = draw.textbbox((0,0), img_id, font=font)
            text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
        text_x = x_offset + (im.width - text_w) // 2
        text_y = max_height + (subtitle_height - text_h) // 2
        draw.text((text_x, text_y), img_id, fill=(0, 0, 0), font=font)
        x_offset += im.width

    # Save combined image
    img_out_path = out_dir / f"example_{idx}.png"
    canvas.save(img_out_path)

    # Collect and write text descriptions
    txt_out_path = out_dir / f"example_{idx}.txt"
    with open(txt_out_path, 'w') as txt_out:
        for img_id, _ in images:
            tag_file = txt_dir / f"{img_id}.txt"
            if tag_file.exists():
                with open(tag_file, 'r') as tf:
                    desc = tf.read().strip()
            else:
                raise FileNotFoundError(f"Tag file not found: {tag_file}")
            txt_out.write(f"ID: {img_id}, description: {desc}\n\n")

print(f"Generated {len(entries)} example image/text pairs in '{out_dir.resolve()}'")



Generated 100 example image/text pairs in '/storage/ice1/1/0/hzhang931/planscape/examples/complex/examples_output'
